# Context Management for AI Agents

This notebook is the runnable companion to [`context_management.md`](./context_management.md). It demonstrates all nine context management strategies using **IBM Granite** on watsonx.ai so you can see real LLM behaviour — not mocked output — at each step.

The context window is an agent's entire working memory. Every strategy below addresses one or more of the four failure modes identified in the blog:

| Failure mode | Root cause |
|---|---|
| **Context Poisoning** | Hallucinations re-enter as ground truth |
| **Context Distraction** | Too much history overwhelms the model |
| **Context Confusion** | Irrelevant tokens pull the model off-target |
| **Context Clash** | Contradictory information accumulates |

## Strategies covered

1. FIFO / Rolling Window — *Distraction*
2. Compaction / Summarization — *Distraction*
3. Dynamic Tool Selection — *Confusion*
4. Context Pruning — *Confusion, Poisoning*
5. Structured Note-Taking (Context Offloading) — *Distraction, Clash*
6. Filesystem as Scratchpad — *Distraction, Clash*
7. Semantic Compression — *Distraction, Confusion*
8. RAG / Vector Retrieval — *Confusion*
9. Sub-agent Architectures (Context Quarantine) — *Clash, Distraction*


## 1. Setup & Installation

You can run this notebook in [Colab](https://colab.research.google.com/) or locally. To avoid Python package conflicts, we recommend a [virtual environment](https://docs.python.org/3/library/venv.html).

### Install dependencies


In [1]:
%pip install -q \
    "git+https://github.com/ibm-granite-community/utils.git" \
    langgraph \
    langchain \
    langchain-core \
    langchain_ibm \
    sentence-transformers \
    numpy


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Connect to the Granite model on watsonx.ai

See [Getting Started with IBM watsonx](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Getting_Started/Getting_Started_with_WatsonX.ipynb) for setup instructions.

You need these environment variables (in a `.env` file or exported in your shell):
- `WATSONX_URL` — your watsonx.ai endpoint
- `WATSONX_APIKEY` — your API key
- `WATSONX_PROJECT_ID` — your project ID


In [2]:
from langchain.chat_models import init_chat_model
from langchain_core.utils.utils import convert_to_secret_str
from ibm_granite_community.notebook_utils import get_env_var

MODEL_ID = "ibm/granite-4-h-small"
# MODEL_ID = "meta-llama/llama-3-3-70b-instruct"

llm = init_chat_model(
    model=MODEL_ID,
    model_provider="ibm",
    url=convert_to_secret_str(get_env_var("WATSONX_URL")),
    apikey=convert_to_secret_str(get_env_var("WATSONX_APIKEY")),
    project_id=get_env_var("WATSONX_PROJECT_ID"),
    params={"temperature": 0, "max_new_tokens": 512},
)

# Quick sanity check
from langchain_core.messages import HumanMessage
ping = llm.invoke([HumanMessage(content="Reply with exactly: ready")])
print("Model response:", ping.content)

WATSONX_URL loaded from .env file.
Model response: ready


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/ibm_watsonx_ai/wml_resource.py:100: WatsonxAPIWarning: The value of 'max_tokens' for this model was set to value 1024
ID: unspecified_max_token
  warn(cls._build_warning_message(warning), WatsonxAPIWarning)


### Shared imports


In [3]:
from __future__ import annotations

import json
import math
import re
import shutil
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path
from pprint import pprint
from typing import Annotated, TypedDict

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

import warnings
from ibm_watsonx_ai.wml_resource import WatsonxAPIWarning



# ---------------------------------------------------------------------------
# Lightweight text helpers used by strategies that do NOT need an LLM
# ---------------------------------------------------------------------------

def simple_tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text.lower())


def keyword_score(query: str, text: str) -> int:
    query_terms = set(simple_tokenize(query))
    text_terms  = set(simple_tokenize(text))
    return len(query_terms & text_terms)


def cosine_similarity(query: str, text: str) -> float:
    left  = Counter(simple_tokenize(query))
    right = Counter(simple_tokenize(text))
    shared    = set(left) & set(right)
    numerator = sum(left[t] * right[t] for t in shared)
    left_norm  = math.sqrt(sum(v * v for v in left.values()))
    right_norm = math.sqrt(sum(v * v for v in right.values()))
    if not left_norm or not right_norm:
        return 0.0
    return numerator / (left_norm * right_norm)

warnings.filterwarnings("ignore", category=WatsonxAPIWarning)


print("Imports OK")

Imports OK



## 2. Strategy 1 — FIFO / Rolling Window

**Goal:** Keep the active prompt small by capping history length to the most recent $N$ messages.

**How it works:** Standard chat models remember past context because we resend the message history with every new turn. A Rolling Window (First-In, First-Out) keeps only the system prompt plus the last $N$ messages. Older turns are automatically dropped off the conveyor belt.Let's define a helper function `apply_fifo_window()` that preserves our `SystemMessage` (so the model remembers its assigned persona) while trimming the remaining chat history to our desired window size.


### Build the Conversation History & Define FIFO Trimming

In a multi-turn conversation, past messages accumulate quickly. First, we generate a multi-turn conversation with Granite about planning a trip to Tokyo. Then, we write a `apply_fifo_window()` function that keeps the system prompt intact while capping the active chat history to only the $N$ most recent turns.

In [4]:
# =====================================================================
# Strategy 1 — FIFO / Rolling Window (Part 1: Build & Define)
# =====================================================================

from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# 1. Compact helper function to print message history cleanly
def print_compact_conversation(messages: list, title: str = "CONVERSATION HISTORY"):
    print(f"\n--- {title} ({len(messages)} messages) ---")
    for idx, msg in enumerate(messages, 1):
        if isinstance(msg, SystemMessage):
            tag = "SYS"
        elif isinstance(msg, HumanMessage):
            tag = "USER"
        else:
            tag = "BOT"
            
        # Clean up inner newlines so each turn stays on a single line
        clean_text = msg.content.replace("\n", " ").strip()
        print(f"{idx}. [{tag}]: {clean_text}")


# 2. Set up persona and user turns
system_prompt = SystemMessage(content="You are a helpful travel planning assistant. Answer concisely.")

turns = [
    "I want to plan a 7-day trip to Tokyo. What season do you recommend?",
    "What is the typical budget range per day for a mid-range traveller?",
    "Which neighbourhoods are best for first-time visitors?",
    "Can you suggest two must-see cultural sites?",
    "What is the easiest way to get from Narita airport to central Tokyo?",
]


# 3. Generate multi-turn history with IBM Granite
conversation: list = [system_prompt]

for user_text in turns:
    conversation.append(HumanMessage(content=user_text))
    response = llm.invoke(conversation)
    conversation.append(AIMessage(content=response.content))


# 4. Define the FIFO (Rolling Window) trimming logic
def apply_fifo_window(messages: list, window_size: int = 3) -> list:
    """
    Preserves the SystemMessage at index 0 (if present)
    and keeps only the last `window_size` messages.
    """
    if not messages:
        return []

    if isinstance(messages[0], SystemMessage):
        system_msg = [messages[0]]
        chat_history = messages[1:]
    else:
        system_msg = []
        chat_history = messages

    trimmed_chat = chat_history[-window_size:]
    return system_msg + trimmed_chat


# 5. Display the full generated conversation
print_compact_conversation(conversation, title="FULL CONVERSATION HISTORY GENERATED WITH GRANITE")
print("\nFIFO Trimming function ready.")


--- FULL CONVERSATION HISTORY GENERATED WITH GRANITE (11 messages) ---
1. [SYS]: You are a helpful travel planning assistant. Answer concisely.
2. [USER]: I want to plan a 7-day trip to Tokyo. What season do you recommend?
3. [BOT]: Spring (March to May) or Autumn (September to November) are recommended for mild weather and beautiful scenery.
4. [USER]: What is the typical budget range per day for a mid-range traveller?
5. [BOT]: A typical budget range per day for a mid-range traveler in Tokyo is around $100 to $200.
6. [USER]: Which neighbourhoods are best for first-time visitors?
7. [BOT]: Shibuya, Shinjuku, Asakusa, and Ginza are great for first-time visitors.
8. [USER]: Can you suggest two must-see cultural sites?
9. [BOT]: Two must-see cultural sites are Senso-ji Temple in Asakusa and Meiji Shrine in Shibuya.
10. [USER]: What is the easiest way to get from Narita airport to central Tokyo?
11. [BOT]: The easiest way is by taking the Narita Express (N'EX) train directly to Tokyo St

### Inspect the Trimmed Context Window

Now we pass our 11-message conversation through `apply_fifo_window()` with `window_size = 3`. This shows  how the window drops older context (like budget and season discussions) while retaining the system prompt and the latest messages.

In [5]:
# --- Part 2: Inspect Context Before vs. After Trimming ---

WINDOW_SIZE = 3

# Apply FIFO trimming
active_context = apply_fifo_window(conversation, window_size=WINDOW_SIZE)

print(f"Original History Length : {len(conversation)} messages")
print(f"Active FIFO Window Size : {len(active_context)} messages\n")

print("--- ACTIVE CONTEXT SENT TO GRANITE ---")
for idx, msg in enumerate(active_context):
    role = "SYS" if isinstance(msg, SystemMessage) else ("USER" if isinstance(msg, HumanMessage) else "BOT")
    print(f"{idx+1}. [{role}]: {msg.content}")

Original History Length : 11 messages
Active FIFO Window Size : 4 messages

--- ACTIVE CONTEXT SENT TO GRANITE ---
1. [SYS]: You are a helpful travel planning assistant. Answer concisely.
2. [BOT]: Two must-see cultural sites are Senso-ji Temple in Asakusa and Meiji Shrine in Shibuya.
3. [USER]: What is the easiest way to get from Narita airport to central Tokyo?
4. [BOT]: The easiest way is by taking the Narita Express (N'EX) train directly to Tokyo Station.


### Test In-Window vs. Out-of-Window Memory Retention

To prove how the FIFO strategy behaves in practice, we ask Granite two follow-up questions:

- In-Window Question: A query about airport transit (which is inside the last 3 turns). Granite will answer accurately.

- Out-of-Window Question: A query referencing the $100–$200 daily budget (which was dropped). Granite will fail to remember the budget, clearly demonstrating the trade-off of rolling windows.

In [6]:
# --- Part 3: Memory Retention Test ---

# 1. In-Window Test (Narita Airport info is present in the active window)
q_in = HumanMessage(content="Can you remind me what train option you suggested for Narita airport?")
test_context_in = active_context + [q_in]
res_in = llm.invoke(test_context_in)

print("=== 1. IN-WINDOW TEST ===")
print(f"User Question: {q_in.content}")
print(f"Granite      : {res_in.content}\n")

# 2. Out-of-Window Test (Budget details were dropped by the rolling window)
q_out = HumanMessage(content="Is a $250 hotel near Shinjuku within the daily budget I mentioned earlier?")
test_context_out = active_context + [q_out]
res_out = llm.invoke(test_context_out)

print("=== 2. OUT-OF-WINDOW TEST ===")
print(f"User Question: {q_out.content}")
print(f"Granite      : {res_out.content}")

=== 1. IN-WINDOW TEST ===
User Question: Can you remind me what train option you suggested for Narita airport?
Granite      : I suggested taking the Narita Express (N'EX) train directly to Tokyo Station.

=== 2. OUT-OF-WINDOW TEST ===
User Question: Is a $250 hotel near Shinjuku within the daily budget I mentioned earlier?
Granite      : A $250 hotel near Shinjuku is generally within a daily budget, but it depends on your other expenses.


Notice how Granite gave a generic guess about what is *"reasonable for Tokyo"* rather than confirming whether $250 fits your specific budget range ($100–$200). Because the budget conversation occurred in turns 4 and 5 (which were trimmed out of the context window), Granite had zero memory that you previously specified a $100–$200 budget.


## 3. Strategy 2 — Compaction / Summarization

**Goal:** Reduce token size without losing important facts from early in the conversation.

**How it works:** Unlike FIFO (which permanently deletes old messages), Compaction takes older messages, feeds them to Granite to generate a concise summary, and then replaces those raw turns with a single Conversation Summary block.

The new prompt structure sent to the LLM becomes:

- System Prompt (Persona & role)
- Summary of Older Turns (Preserves budget, goals, key details)
- Recent Raw Turns (Keeps exact phrasing for current discussion)

### Defining the LLM-Driven Compaction Logic

In this cell, we write a function that splits the conversation into "old" and "recent" turns. It calls Granite to summarize the old turns into bullet points, then stitches everything back together.

In [7]:
# =====================================================================
# Strategy 2 — Compaction / Summarization (Part 1: Define Compactor)
# =====================================================================

from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

def compact_conversation_history(messages: list, recent_turns_to_keep: int = 2) -> list:
    """
    Summarizes older messages using Granite and returns a compacted message list.
    """
    if not messages:
        return []

    # 1. Separate system message and chat history
    if isinstance(messages[0], SystemMessage):
        system_msg = [messages[0]]
        chat_history = messages[1:]
    else:
        system_msg = []
        chat_history = messages

    # If the history is short, no need to compact
    if len(chat_history) <= recent_turns_to_keep:
        return messages

    # 2. Split history into older turns (to compact) and recent turns (to keep raw)
    old_turns = chat_history[:-recent_turns_to_keep]
    recent_turns = chat_history[-recent_turns_to_keep:]

    # Format older turns into text for the summarization prompt
    formatted_old_turns = ""
    for msg in old_turns:
        role = "User" if isinstance(msg, HumanMessage) else "Assistant"
        formatted_old_turns += f"{role}: {msg.content}\n"

    # 3. Ask Granite to condense the old turns into key facts
    summary_prompt = [
        SystemMessage(content="You are a precise conversation summarizer."),
        HumanMessage(content=(
            "Summarize the key decisions, constraints, user preferences, and facts from "
            "this conversation history into 3-4 bullet points. Be concise.\n\n"
            f"Conversation History:\n{formatted_old_turns}"
        ))
    ]
    
    summary_response = llm.invoke(summary_prompt)
    summary_text = summary_response.content.strip()

    # 4. Construct compacted state: System Message + Summary + Recent Turns
    compacted_summary_msg = SystemMessage(
        content=f"--- SUMMARY OF EARLIER CONVERSATION ---\n{summary_text}"
    )

    return system_msg + [compacted_summary_msg] + recent_turns

print("Compaction function ready.")

Compaction function ready.


### Inspecting the Compacted Context

Let's take our 11-message conversation from Strategy 1 and pass it through compact_conversation_history(), keeping only the 2 most recent raw turns.

Granite will condense turns 1 through 7 into a short summary block while keeping turns 8 through 11 in their original raw text.

In [8]:
# =====================================================================
# Strategy 2 — Compaction / Summarization (Part 2: Run Compaction)
# =====================================================================

# Compact our 11-message conversation from Strategy 1
compacted_context = compact_conversation_history(conversation, recent_turns_to_keep=4)

print(f"Original History Length : {len(conversation)} messages")
print(f"Compacted Context Size  : {len(compacted_context)} messages\n")

print_compact_conversation(compacted_context, title="COMPACTED CONTEXT SENT TO GRANITE")

Original History Length : 11 messages
Compacted Context Size  : 6 messages


--- COMPACTED CONTEXT SENT TO GRANITE (6 messages) ---
1. [SYS]: You are a helpful travel planning assistant. Answer concisely.
2. [SYS]: --- SUMMARY OF EARLIER CONVERSATION --- - Recommended seasons for a 7-day Tokyo trip: Spring (March to May) or Autumn (September to November) for mild weather and beautiful scenery. - Typical daily budget range for a mid-range traveler in Tokyo: $100 to $200. - Best neighbourhoods for first-time visitors: Shibuya, Shinjuku, Asakusa, and Ginza. - User plans a 7-day trip to Tokyo and seeks recommendations for season, budget, and neighbourhoods.
3. [USER]: Can you suggest two must-see cultural sites?
4. [BOT]: Two must-see cultural sites are Senso-ji Temple in Asakusa and Meiji Shrine in Shibuya.
5. [USER]: What is the easiest way to get from Narita airport to central Tokyo?
6. [BOT]: The easiest way is by taking the Narita Express (N'EX) train directly to Tokyo Station.


### Testing Memory Retention After Compaction

Recall that under FIFO (Strategy 1), Granite forgot our $100–$200 daily budget because those turns were dropped.

Let's ask the exact same question again using our `compacted_context` to see if the summary successfully preserved that information!

In [9]:
# =====================================================================
# Strategy 2 — Compaction / Summarization (Part 3: Verify Memory)
# =====================================================================

# Ask the budget question that failed under FIFO
budget_query = HumanMessage(
    content="Is a $250 per night hotel near Shinjuku within the daily budget I mentioned earlier?"
)

# Append the question to the compacted context
test_context = compacted_context + [budget_query]
response = llm.invoke(test_context)

print("=== COMPACTION MEMORY TEST ===")
print(f"User Question: {budget_query.content}\n")
print(f"Granite Response:\n{response.content}")

=== COMPACTION MEMORY TEST ===
User Question: Is a $250 per night hotel near Shinjuku within the daily budget I mentioned earlier?

Granite Response:
No, a $250 per night hotel exceeds your daily budget of $100 to $200.


Under FIFO (Strategy 1), Granite failed this exact budget check because the budget turns were completely deleted from the active prompt.

Under Compaction (Strategy 2), Granite correctly recognized that $250 exceeds your $100–$200 limit. In this run, the summary preserved the budget constraint even though the original turn was removed from the raw history. That demonstrates the value of compaction: it can retain important facts that FIFO would lose while still reducing prompt size. But compaction is still lossy overall, and a weaker summary could omit or distort important details.

## 4. Strategy 3 — Dynamic Tool Selection

**Goal:** Prevent tool bloat and context confusion by dynamically retrieving and binding only relevant executable tools to the LLM.

**How it works:** Having 10+ tool schemas bound to an agent wastes tokens and degrades selection accuracy. Here, we define real, callable tools, index their descriptions into a vector space, retrieve the top $K$ matching tool objects, and use `llm.bind_tools()` to make them natively executable by the LLM.

### Defining Executable Tools and Building the Vector Index

In [10]:
# =====================================================================
# Strategy 3 — Dynamic Tool Selection (Part 1: Real Tools & Index)
# =====================================================================

import numpy as np
from langchain_core.tools import tool
from sentence_transformers import SentenceTransformer

# 1. Define actual callable LangChain tools using the @tool decorator
@tool
def weather_lookup(city: str) -> str:
    """Get current weather conditions and 7-day forecast for any city."""
    return f"Weather for {city}: Partly cloudy, 22°C, 10% chance of rain."

@tool
def calendar_reader(query: str) -> str:
    """Read meeting schedules, check calendar availability, and find open time slots."""
    return "Friday Schedule: Outdoor team meeting scheduled at 2:00 PM."

@tool
def sql_runner(query: str) -> str:
    """Execute SQL queries against structured corporate sales and revenue databases."""
    return "Database result: Total Q3 Revenue = $1.2M."

@tool
def docs_search(query: str) -> str:
    """Search internal company technical documentation, policy guides, and runbooks."""
    return "Doc result: Found 3 relevant architecture pages."

@tool
def flight_search(origin: str, destination: str) -> str:
    """Find commercial flight options, compare ticket fares, and check flight status."""
    return f"Flights from {origin} to {destination}: 3 direct flights available."

@tool
def calculator(expression: str) -> str:
    """Perform complex mathematical operations, financial projections, and conversions."""
    return "Calculated result: 42."

@tool
def email_sender(recipient: str, subject: str, body: str) -> str:
    """Draft and send corporate emails or notifications to team members."""
    return f"Email sent successfully to {recipient}."

@tool
def jira_manager(action: str, issue_id: str) -> str:
    """Create, update, or query software development tickets and bug reports."""
    return f"Jira ticket {issue_id} updated."

@tool
def hotel_booking(city: str, price_limit: int) -> str:
    """Search for hotel rooms, check room rates, and make lodging reservations."""
    return f"Found 5 hotel options in {city} under ${price_limit}/night."

@tool
def currency_converter(amount: float, from_curr: str, to_curr: str) -> str:
    """Convert monetary amounts between foreign currencies using live exchange rates."""
    return f"{amount} {from_curr} = {amount * 0.92} {to_curr}."


# 2. Store tools in a master dictionary
tool_registry = [
    weather_lookup, calendar_reader, sql_runner, docs_search, flight_search,
    calculator, email_sender, jira_manager, hotel_booking, currency_converter
]

# 3. Load embedding model and index tool descriptions
embedder = SentenceTransformer("all-MiniLM-L6-v2")
tool_descriptions = [t.description for t in tool_registry]
tool_embeddings = embedder.encode(tool_descriptions, convert_to_numpy=True)

print(f"Tool Registry Ready: {len(tool_registry)} real executable tools indexed.")

Tool Registry Ready: 10 real executable tools indexed.


### Creating the Dynamic Tool Retriever

In [11]:
# =====================================================================
# Strategy 3 — Dynamic Tool Selection (Part 2: Dynamic Retriever)
# =====================================================================

def retrieve_relevant_tools(user_query: str, top_k: int = 2) -> list:
    """
    Encodes query and returns top_k matching tool objects.
    """
    query_embedding = embedder.encode(user_query, convert_to_numpy=True)

    # Cosine similarity matching
    scores = np.dot(tool_embeddings, query_embedding) / (
        np.linalg.norm(tool_embeddings, axis=1) * np.linalg.norm(query_embedding)
    )

    ranked_indices = np.argsort(scores)[::-1][:top_k]

    selected_tools = []
    print(f"\n--- DYNAMIC TOOL RETRIEVAL FOR QUERY: '{user_query}' ---")
    for idx in ranked_indices:
        tool_obj = tool_registry[idx]
        score = scores[idx]
        print(f"  • Match Score: {score:.3f} | Tool: {tool_obj.name:<18} -> {tool_obj.description}")
        selected_tools.append(tool_obj)

    return selected_tools

print("Tool retriever ready.")

Tool retriever ready.


### Dynamically Binding Tools & Executing the Call

In [12]:
# =====================================================================
# Strategy 3 — Dynamic Tool Selection (Part 3: Binding & Execution)
# =====================================================================

from langchain_core.messages import HumanMessage

# User query needing specific tools
user_query = "Can you check if it is raining in Tokyo on Friday during my outdoor team meeting?"

# 1. Dynamically retrieve top 2 tool objects
filtered_tools = retrieve_relevant_tools(user_query, top_k=2)

# 2. Dynamically BIND only the retrieved tools to the model
llm_with_tools = llm.bind_tools(filtered_tools)

# 3. Invoke model to generate native structured tool calls
response = llm_with_tools.invoke([HumanMessage(content=user_query)])

print("\n=== MODEL OUTPUT WITH BOUND TOOLS ===")
print("Bound Tools Count:", len(filtered_tools))

# Check and print the generated tool calls
if response.tool_calls:
    print("\n✓ Native Tool Calls Generated:")
    for call in response.tool_calls:
        print(f"  - Calling Tool: {call['name']}(args={call['args']})")
else:
    print("\nModel Response:")
    print(response.content)


--- DYNAMIC TOOL RETRIEVAL FOR QUERY: 'Can you check if it is raining in Tokyo on Friday during my outdoor team meeting?' ---
  • Match Score: 0.328 | Tool: weather_lookup     -> Get current weather conditions and 7-day forecast for any city.
  • Match Score: 0.261 | Tool: calendar_reader    -> Read meeting schedules, check calendar availability, and find open time slots.

=== MODEL OUTPUT WITH BOUND TOOLS ===
Bound Tools Count: 2

✓ Native Tool Calls Generated:
  - Calling Tool: weather_lookup(args={'city': 'Tokyo'})


As you read the output closely you can see two separate effects: 
- Out of 10 available tools, the retriever narrowed the available set to the 2 most relevant tools (`weather_lookup` and `calendar_reader`) for this query.
- The model then generated a structured call to `weather_lookup(args={'city': 'Tokyo'})`. In other words, retrieval successfully reduced tool bloat, but the model did not choose to call every relevant tool it had available.

**Key Takeaway:** Dynamic tool selection helps by reducing prompt clutter and limiting the action space the model sees. It improves the odds of correct tool use, but it does not guarantee perfect multi-tool planning on every run.

## 5. Strategy 4 — Context Pruning

**Goal:** Eliminate noisy, irrelevant log entries and distractors before they poison the agent's context window.

**How it works:** When agents query monitoring tools or APIs, they receive walls of raw text containing high-noise data. Unpruned context causes Context Poisoning, leading the model to hallucinate or summarize irrelevant distractors.

Unlike summarization (which rephrases text), pruning acts as a surgical filter—discarding distracting lines completely while preserving the exact raw text of relevant entries.


### Setting Up Noisy Tool Stream with a Target Signal

In [13]:
# =====================================================================
# Strategy 4 — Context Pruning (Part 1: Realistic Noisy Log Stream)
# =====================================================================

# Simulated raw monitoring log output containing heavy noise + 1 critical signal
raw_tool_output = """
[13:58:01] SYSTEM: Routine database backup completed. 0 errors logged.
[13:59:12] DEPLOY: Automated frontend asset sync to CDN region US-East finished.
[14:01:05] ALERT: Customer checkout API returned 500 errors. Cause: DB connection pool exhausted (max 50/50 connections occupied) due to unindexed query on 'orders_v2' table.
[14:02:30] MEMO: Reminder - Engineering team brown-bag lunch session starts at 12:30 PM tomorrow in Room 4B.
[14:03:15] INFRA: Kubernetes node worker-node-8 autoscaled CPU from 40% to 55%.
[14:04:22] SECURITY: Automated SSL certificate renewal check succeeded for domain checkout.internal.
[14:05:00] LOG: Internal admin dashboard user logged out after 30 mins idle time.
[14:06:10] DEPLOY: CI/CD pipeline #8841 status changed to SUCCESS for microservice-billing.
""".strip()

# Target query requiring a specific answer buried in the noise
user_query = "What specific root cause caused the customer checkout failures during the 14:00 window?"

print(f"Raw Monitoring Stream Loaded ({len(raw_tool_output)} characters).")
print(f"Target Query: '{user_query}'\n")
print("--- RAW UNPRUNED LOG CONTEXT ---")
print(raw_tool_output[:300] + "\n... [Rest of raw logs omitted] ...")

Raw Monitoring Stream Loaded (790 characters).
Target Query: 'What specific root cause caused the customer checkout failures during the 14:00 window?'

--- RAW UNPRUNED LOG CONTEXT ---
[13:58:01] SYSTEM: Routine database backup completed. 0 errors logged.
[13:59:12] DEPLOY: Automated frontend asset sync to CDN region US-East finished.
[14:01:05] ALERT: Customer checkout API returned 500 errors. Cause: DB connection pool exhausted (max 50/50 connections occupied) due to unindexed q
... [Rest of raw logs omitted] ...


### Defining the Surgical Pruner Node

In [14]:
# =====================================================================
# Strategy 4 — Context Pruning (Part 2: Pruner Logic)
# =====================================================================

from langchain_core.messages import HumanMessage, SystemMessage

def prune_context_with_llm(query: str, raw_context: str) -> str:
    """
    Surgically extracts only raw log lines relevant to the target query.
    """
    pruning_prompt = [
        SystemMessage(content=(
            "You are a precise log filter. Extract ONLY the exact log lines or passages "
            "from the raw input that directly relate to or answer the user's target query.\n"
            "- Do NOT rephrase, summarize, or alter the original log text.\n"
            "- Do NOT add conversational intro or outro filler.\n"
            "- Completely remove all unrelated system logs, deploys, and announcements."
        )),
        HumanMessage(content=(
            f"TARGET QUERY: {query}\n\n"
            f"RAW LOGS TO PRUNE:\n{raw_context}"
        ))
    ]
    
    response = llm.invoke(pruning_prompt)
    return response.content.strip()

print("Context Pruner node ready.")

Context Pruner node ready.


### Comparing Results (Without Pruning vs. With Pruning)

In [15]:
# =====================================================================
# Strategy 4 — Context Pruning (Part 3: Before vs After Comparison)
# =====================================================================

# 1. Execute pruning to isolate signal from noise
pruned_context = prune_context_with_llm(user_query, raw_tool_output)

# 2. Call Agent WITHOUT Pruning (Unfiltered noisy logs)
prompt_unpruned = [
    SystemMessage(content="You are a SRE Incident Analyst. Answer the user's query directly based on the context."),
    HumanMessage(content=f"QUERY: {user_query}\n\nCONTEXT:\n{raw_tool_output}")
]
response_unpruned = llm.invoke(prompt_unpruned)

# 3. Call Agent WITH Pruning (Clean filtered signal)
prompt_pruned = [
    SystemMessage(content="You are a SRE Incident Analyst. Answer the user's query directly based on the context."),
    HumanMessage(content=f"QUERY: {user_query}\n\nCONTEXT:\n{pruned_context}")
]
response_pruned = llm.invoke(prompt_pruned)


# --- DISPLAY COMPARISON METRICS & RESPONSES ---
print("=" * 70)
print(" 1. CONTEXT FOOTPRINT METRICS")
print("=" * 70)
print(f"Unpruned Context Size : {len(raw_tool_output)} characters")
print(f"Pruned Context Size   : {len(pruned_context)} characters")
print(f"Noise Reduction       : {((len(raw_tool_output) - len(pruned_context)) / len(raw_tool_output)) * 100:.1f}% removed")

print("\n" + "=" * 70)
print(" 2. PRUNED CONTEXT PASSED TO AGENT")
print("=" * 70)
print(pruned_context)

print("\n" + "=" * 70)
print(" 3. AGENT RESPONSE COMPARISON")
print("=" * 70)
print(f"--- Response WITHOUT Pruning (Full Noisy Log) ---\n{response_unpruned.content}\n")
print(f"--- Response WITH Pruning (Pruned Log) ---\n{response_pruned.content}")

 1. CONTEXT FOOTPRINT METRICS
Unpruned Context Size : 790 characters
Pruned Context Size   : 174 characters
Noise Reduction       : 78.0% removed

 2. PRUNED CONTEXT PASSED TO AGENT
[14:01:05] ALERT: Customer checkout API returned 500 errors. Cause: DB connection pool exhausted (max 50/50 connections occupied) due to unindexed query on 'orders_v2' table.

 3. AGENT RESPONSE COMPARISON
--- Response WITHOUT Pruning (Full Noisy Log) ---
Based on the provided context, the specific root cause of the customer checkout failures during the 14:00 window was an exhausted database connection pool (max 50/50 connections occupied) due to an unindexed query on the 'orders_v2' table. This issue was triggered by the automated frontend asset sync to the CDN region US-East that finished at 13:59:12.

--- Response WITH Pruning (Pruned Log) ---
Based on the provided context, the specific root cause of the customer checkout failures during the 14:00 window was an exhausted database (DB) connection pool. Th

As reflected in **CONTEXT FOOTPRINT METRICS**, the pruner stripped out 78% of the clutter (unrelated CDN syncs, brown-bag lunch memos, SSL renewals) and isolated the exact alert log most relevant to the issue.

Look at the *Unpruned Response*: the model got distracted by irrelevant noise and falsely claimed the database failure was *"triggered by the automated frontend asset sync."*

In the *Pruned Response*, removing the surrounding noise made it much easier for the agent to ground its answer in the real signal: an unindexed query on `orders_v2` exhausting the connection pool. The extra troubleshooting steps are still generated analysis, but they are now based on cleaner evidence.

Context Pruning is not just about saving tokens. It reduces the chance that noisy tool output will pull the model toward false correlations or irrelevant details.

## 6. Strategy 5 — Structured Note-Taking (Context Offloading)

**Goal:** Keep the active conversation history lean by offloading durable facts and structured knowledge into LangGraph state.

**How it works:** Instead of keeping every raw search result or tool output in the chat history, the agent uses an extraction step to write key findings into a structured dictionary (NoteState). The raw messages can then be discarded, while the agent's "working memory" remains clean and structured.


### Defining the LangGraph State and Research Data

In [16]:
# =====================================================================
# Strategy 5 — Structured Note-Taking (Part 1: State & Setup)
# =====================================================================

from typing import TypedDict, Annotated
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langgraph.graph import StateGraph, START, END

# 1. Define a LangGraph State that holds structured notes separately from messages
class NoteAgentState(TypedDict):
    raw_findings: list[str]          # Temporary raw search/tool findings
    structured_notes: dict[str, str] # Durable offloaded state (Key-Value)
    final_summary: str              # Final output generated from notes


# 2. Simulated multi-step research findings from an RFP evaluation
rfp_research_findings = [
    "Legal analysis: The vendor agreement mandates strict EU GDPR compliance with data residency in Frankfurt.",
    "Finance review: Maximum annual software license budget is capped at $120,000 USD with net-30 payment terms.",
    "Security audit: Vendor must hold active SOC2 Type II certification and support SSO via SAML 2.0.",
]

print("LangGraph NoteAgentState schema ready.")

LangGraph NoteAgentState schema ready.


### Building the LangGraph Extraction & Offloading Nodes

We create two LangGraph nodes:

- `extract_notes_node:` Parses raw findings and writes clean, key-value notes into structured_notes using the LLM.
- `synthesize_from_notes_node:` Writes a final report using only the structured_notes dictionary without needing any past chat turns!

In [17]:
# =====================================================================
# Strategy 5 — Structured Note-Taking (Part 2: LangGraph Nodes)
# =====================================================================

import json

# Node 1: Extract structured key-value pairs from raw findings
def extract_notes_node(state: NoteAgentState) -> NoteAgentState:
    existing_notes = dict(state.get("structured_notes", {}))
    raw_findings = state.get("raw_findings", [])
    
    prompt = [
        SystemMessage(content=(
            "You are a structured note-taking assistant. Extract core facts from the provided text.\n"
            "Return a valid JSON object where keys represent topic categories (e.g., 'compliance_requirement', 'budget_cap') "
            "and values are concise statement facts. Output ONLY valid JSON."
        )),
        HumanMessage(content=f"Raw Findings to Offload:\n" + "\n".join(raw_findings))
    ]
    
    response = llm.invoke(prompt)
    
    # Clean and parse JSON response
    try:
        clean_json_str = response.content.strip().replace("```json", "").replace("```", "").strip()
        new_notes = json.loads(clean_json_str)
        existing_notes.update(new_notes)
    except Exception:
        # Fallback if LLM returns plain text
        existing_notes["extracted_summary"] = response.content.strip()
        
    return {**state, "structured_notes": existing_notes}


# Node 2: Synthesize final output strictly from the offloaded notes
def synthesize_from_notes_node(state: NoteAgentState) -> NoteAgentState:
    notes = state.get("structured_notes", {})
    
    # Format notes cleanly for the LLM
    formatted_notes = json.dumps(notes, indent=2)
    
    prompt = [
        SystemMessage(content="You are an executive summary writer. Synthesize the structured notes into an RFP evaluation brief."),
        HumanMessage(content=f"STRUCTURED NOTES (OFFLOADED CONTEXT):\n{formatted_notes}")
    ]
    
    response = llm.invoke(prompt)
    return {**state, "final_summary": response.content.strip()}


# Build the StateGraph
workflow = StateGraph(NoteAgentState)
workflow.add_node("extract_notes", extract_notes_node)
workflow.add_node("synthesize", synthesize_from_notes_node)

workflow.add_edge(START, "extract_notes")
workflow.add_edge("extract_notes", "synthesize")
workflow.add_edge("synthesize", END)

note_graph = workflow.compile()
print("Context Offloading LangGraph workflow compiled successfully.")

Context Offloading LangGraph workflow compiled successfully.


### Executing the Workflow

Let's run the graph with our RFP research findings.

Notice how the state offloads raw text into structured JSON notes, allowing the final synthesis step to execute with zero bloated chat message history.

In [18]:
# =====================================================================
# Strategy 5 — Structured Note-Taking (Part 3: Run & Inspect)
# =====================================================================

# Initial state containing raw findings and an empty notes dictionary
initial_state = {
    "raw_findings": rfp_research_findings,
    "structured_notes": {},
    "final_summary": ""
}

# Run the LangGraph execution
graph_result = note_graph.invoke(initial_state)


# --- DISPLAY RESULTS ---
print("=" * 70)
print(" 1. OFFLOADED STRUCTURED NOTES (LANGGRAPH STATE)")
print("=" * 70)
print(json.dumps(graph_result["structured_notes"], indent=2))

print("\n" + "=" * 70)
print(" 2. FINAL EXECUTIVE BRIEF (SYNTHESIZED FROM NOTES ONLY)")
print("=" * 70)
print(graph_result["final_summary"])

 1. OFFLOADED STRUCTURED NOTES (LANGGRAPH STATE)
{
  "compliance_requirement": "EU GDPR compliance with data residency in Frankfurt",
  "budget_cap": "Maximum annual software license budget is $120,000 USD with net-30 payment terms",
  "security_requirement": "Vendor must hold active SOC2 Type II certification and support SSO via SAML 2.0"
}

 2. FINAL EXECUTIVE BRIEF (SYNTHESIZED FROM NOTES ONLY)
RFP Evaluation Brief

1. Compliance Requirement:
   - The selected vendor must ensure full compliance with the EU General Data Protection Regulation (GDPR).
   - Data residency must be maintained within Frankfurt, Germany, to adhere to local data protection laws and regulations.
   - The vendor should provide documentation and certifications demonstrating their GDPR compliance and data residency practices.

2. Budget Cap:
   - The maximum annual software license budget for this project is $120,000 USD.
   - The vendor's pricing proposal must not exceed this budget limit.
   - Payment terms sh

As you can see in the output, instead of retaining whole paragraphs of findings in the message list, the agent extracted and stored key operational constraints in a structured key-value state object (`compliance_requirement`, `budget_cap`, `security_requirement`).

When composing the final executive brief, the agent referenced these structured facts instead of re-reading the raw findings. That is the core benefit of note-taking as context offloading: important information stays accessible without forcing every earlier message back into the prompt.

Structured Note-Taking helps reduce prompt bloat and can reduce context clash by giving the agent a cleaner working memory. In this notebook, the notes live in LangGraph state for the workflow run; they are structured and useful, but not automatically long-term durable unless you persist them separately.

## 7. Strategy 6 — Filesystem as Scratchpad

**Goal:** Enable persistent, long-term memory across multi-step agent runs by writing state to disk files rather than memory prompts.

**How it works:** When working with large datasets or multi-phase tasks, the agent creates a dedicated scratchpad directory. It writes markdown plans or JSON findings to disk and reads back only the specific files required for the current step.

*Note for Security: Always scope file operations to a isolated workspace folder (e.g., ./tmp/agent_workspace) to prevent unauthorized access to system files.*


### Setting up the Sandboxed Workspace

In [19]:
# =====================================================================
# Strategy 6 — Filesystem as Scratchpad (Part 1: Workspace Setup)
# =====================================================================

import json
import shutil
from pathlib import Path

# 1. Create a sandboxed working directory for our agent
workspace_dir = Path("tmp/agent_scratchpad")

# Clean existing directory if present to ensure a fresh run
if workspace_dir.exists():
    shutil.rmtree(workspace_dir)
workspace_dir.mkdir(parents=True, exist_ok=True)

# 2. Define path constants for persistent files
plan_file = workspace_dir / "incident_plan.md"
findings_file = workspace_dir / "findings.json"

print(f"Sandboxed agent workspace ready at: '{workspace_dir.resolve()}'")

Sandboxed agent workspace ready at: '/Users/vrundagadesha/Documents/GitHub/granite-agent-cookbook/recipes/ContextManagement/tmp/agent_scratchpad'


### Agent Writing to Disk Scratchpad

Now, the agent simulates an incident response step by creating a step-by-step resolution plan `(incident_plan.md)` and saving structured technical telemetry `(findings.json)` directly to local files.

In [20]:
# =====================================================================
# Strategy 6 — Filesystem as Scratchpad (Part 2: Disk Writes)
# =====================================================================

# 1. Write incident response checklist to markdown scratchpad
plan_content = """# INCIDENT RESPONSE PLAN
Status: ACTIVE
1. [DONE] Isolate affected database cluster in US-East region.
2. [IN_PROGRESS] Identify root cause for elevated 504 API gateway timeouts.
3. [PENDING] Draft public status page communication for impacted customers.
"""
plan_file.write_text(plan_content.strip(), encoding="utf-8")

# 2. Write telemetry research payload to JSON scratchpad
findings_data = {
    "incident_id": "INC-8821",
    "affected_service": "Payment Gateway API",
    "root_cause": "Unindexed database query caused connection pool exhaustion during traffic peak",
    "customer_impact": "1.2% of checkout attempts failed between 14:00 and 14:18 UTC",
    "mitigation": "Added missing composite index on orders table; connection pool recovered"
}
findings_file.write_text(json.dumps(findings_data, indent=2), encoding="utf-8")

print(f"✓ Saved: {plan_file.name} ({plan_file.stat().st_size} bytes)")
print(f"✓ Saved: {findings_file.name} ({findings_file.stat().st_size} bytes)")

✓ Saved: incident_plan.md (254 bytes)
✓ Saved: findings.json (351 bytes)


### Selective Reading and Status Generation

Instead of dumping all raw files into memory at once, the agent selectively reads only findings.json and the active tasks from incident_plan.md to compose an executive stakeholder update.

In [21]:
# =====================================================================
# Strategy 6 — Filesystem as Scratchpad (Part 3: Selective Read)
# =====================================================================

from langchain_core.messages import HumanMessage, SystemMessage

# 1. Read scratchpad files selectively from disk
saved_plan = plan_file.read_text(encoding="utf-8")
saved_findings = json.loads(findings_file.read_text(encoding="utf-8"))

# 2. Instruct the LLM using the disk-persisted findings
prompt = [
    SystemMessage(content=(
        "You are an Incident Response Lead. Compose a concise executive status update "
        "using ONLY the provided incident plan and telemetry findings stored in the scratchpad."
    )),
    HumanMessage(content=(
        f"SCRATCHPAD FILE: {plan_file.name}\n{saved_plan}\n\n"
        f"SCRATCHPAD FILE: {findings_file.name}\n{json.dumps(saved_findings, indent=2)}"
    ))
]

response = llm.invoke(prompt)

print("=== EXECUTIVE INCIDENT UPDATE (GENERATED FROM SCRATCHPAD PERSISTENCE) ===")
print(response.content)

=== EXECUTIVE INCIDENT UPDATE (GENERATED FROM SCRATCHPAD PERSISTENCE) ===
Executive Status Update:

The incident response team has successfully isolated the affected database cluster in the US-East region as per the incident response plan. We are currently in the process of identifying the root cause for the elevated 504 API gateway timeouts experienced by our Payment Gateway API service.

Preliminary findings indicate that an unindexed database query led to connection pool exhaustion during a traffic peak, resulting in 1.2% of checkout attempts failing between 14:00 and 14:18 UTC. The team has taken immediate mitigation steps by adding a missing composite index on the orders table, which has successfully recovered the connection pool.

We are actively working on drafting a public status page communication to keep our impacted customers informed about the situation and the steps we are taking to resolve the issue. The incident response team will continue to monitor the situation closel

In this process, rather than relying on the LLM's finite context window to keep track of steps over time, intermediate work (incident plans and findings) was written directly to local disk files (`.md` and `.json`).

The agent then reloaded those scratchpad files to generate an executive status update. This demonstrates the main advantage of filesystem offloading: important state can live outside the prompt and be recovered later instead of being re-sent on every turn.

Using the Filesystem as a Scratchpad helps prevent memory loss during complex, multi-step tasks and can support pause/resume workflows across runs. In this notebook, the example shows persistence to disk, though it reads the saved files back directly rather than using a more selective `grep`-style retrieval pattern.

## 8. Strategy 7 — Semantic Compression

**Goal:** Maximally reduce token overhead by distilling unstructured conversation threads into compact, structured JSON schemas.

**How it works:** Unlike natural language summarization, Semantic Compression extracts core operational constraints, numeric parameters, and status flags into a structured state schema.

*Important Conceptual Distinction:* Semantic compression is inherently lossy. While it preserves key decision parameters (e.g., agreed budgets, deadlines, SLAs) with high fidelity, it intentionally sacrifices conversational nuance, speaker sentiment, contingency context, and unmodeled side facts to achieve maximum token efficiency.

### Loading Unstructured Procurement Thread

In [22]:
# =====================================================================
# Strategy 7 — Semantic Compression (Part 1: Raw Unstructured Thread)
# =====================================================================

# Simulated multi-turn corporate procurement discussion
raw_procurement_thread = """
Vendor Rep (10:00 AM): Hi team, thanks for hopping on the call. Regarding the Enterprise Cloud Storage renewal, 
our base tier starts at $150,000 annually for 500TB with standard 99.9% uptime SLA.

Procurement Lead (10:05 AM): Thanks Bob. That $150k is over our allocated department budget. Our hard cap for this fiscal year is $125,000 total. 
Also, 99.9% uptime isn't sufficient for our core billing infrastructure — we require a strict 99.99% uptime guarantee with financial penalty clauses for downtime.

Vendor Rep (10:15 AM): Understood on the SLA. We can bump you to the 99.99% Enterprise SLA Tier with SLAs backed by credits. 
If we keep storage at 350TB instead of 500TB, we can meet your $125,000 cap on a 2-year contract signed by September 30.

Security Officer (10:20 AM): Before we agree to terms, please confirm that data encryption at rest uses AES-256 and all primary data centers are strictly located within the US-West region for SOC2 compliance.

Vendor Rep (10:25 AM): Confirmed on security! Full AES-256 encryption at rest, US-West region hosting only, and SOC2 Type II reports will be attached as Exhibit B to the final contract.
""".strip()

print(f"Raw Procurement Thread Loaded ({len(raw_procurement_thread)} characters).")

Raw Procurement Thread Loaded (1152 characters).


### Defining the Realistic Semantic Compressor Engine

In [23]:
# =====================================================================
# Strategy 7 — Semantic Compression (Part 2: Compressor Logic)
# =====================================================================

import json
from langchain_core.messages import HumanMessage, SystemMessage

def compress_semantics_to_schema(raw_text: str) -> dict:
    """
    Distills unstructured dialogue into a structured JSON parameter schema.
    """
    compression_prompt = [
        SystemMessage(content=(
            "You are a Semantic Compression Engine for enterprise AI agents.\n"
            "Your task is to distill unstructured procurement dialogue into a compact, structured JSON state object.\n"
            "RULES:\n"
            "1. Extract essential operational parameters: budget_cap, agreed_price, storage_capacity, uptime_sla, "
            "contract_term, signing_deadline, encryption_standard, regional_compliance, security_cert.\n"
            "2. Strip conversational greetings, politeness, speaker names, and filler prose.\n"
            "3. Output ONLY valid JSON."
        )),
        HumanMessage(content=f"UNSTRUCTURED THREAD TO COMPRESS:\n{raw_text}")
    ]
    
    response = llm.invoke(compression_prompt)
    
    # Parse and clean JSON response
    clean_json_str = response.content.strip().replace("```json", "").replace("```", "").strip()
    return json.loads(clean_json_str)

print("Semantic Compressor engine ready.")

Semantic Compressor engine ready.


### Executing Compression & Explicitly Analyzing Information Loss

In [24]:
# =====================================================================
# Strategy 7 — Semantic Compression (Part 3: Run & Analyze Loss)
# =====================================================================

# Run semantic compression
compressed_schema = compress_semantics_to_schema(raw_procurement_thread)
compressed_json_str = json.dumps(compressed_schema, indent=2)

print("=" * 70)
print(" 1. COMPRESSED STRUCTURED STATE SCHEMA")
print("=" * 70)
print(compressed_json_str)

print("\n" + "=" * 70)
print(" 2. TOKEN REDUCTION METRICS")
print("=" * 70)
print(f"Original Raw Thread Size : {len(raw_procurement_thread)} characters")
print(f"Compressed Schema Size   : {len(compressed_json_str)} characters")
print(f"Context Overhead Saved   : {((len(raw_procurement_thread) - len(compressed_json_str)) / len(raw_procurement_thread)) * 100:.1f}% reduced")

print("\n" + "=" * 70)
print(" 3. COMPRESSION LOSS ANALYSIS (PRESERVED vs DISCARDED)")
print("=" * 70)
print("✓ PRESERVED (Hard Decision Parameters):")
print("  - Pricing ($125k), Storage (350TB), SLA (99.99%), Deadline (Sept 30), Security (AES-256/SOC2)")
print("\n✗ DISCARDED (Lossy Context Trade-offs):")
print("  - Negotiation history & posture (Vendor's initial $150k asking price & 500TB offer)")
print("  - Specific contingency mechanics (Financial penalty clauses & SLA credits details)")
print("  - Speaker identity, tone, and evidentiary transcript traceability")

 1. COMPRESSED STRUCTURED STATE SCHEMA
{
  "budget_cap": 125000,
  "agreed_price": 125000,
  "storage_capacity": 350,
  "uptime_sla": "99.99%",
  "contract_term": "2-year",
  "signing_deadline": "September 30",
  "encryption_standard": "AES-256",
  "regional_compliance": "US-West",
  "security_cert": "SOC2 Type II"
}

 2. TOKEN REDUCTION METRICS
Original Raw Thread Size : 1152 characters
Compressed Schema Size   : 279 characters
Context Overhead Saved   : 75.8% reduced

 3. COMPRESSION LOSS ANALYSIS (PRESERVED vs DISCARDED)
✓ PRESERVED (Hard Decision Parameters):
  - Pricing ($125k), Storage (350TB), SLA (99.99%), Deadline (Sept 30), Security (AES-256/SOC2)

✗ DISCARDED (Lossy Context Trade-offs):
  - Negotiation history & posture (Vendor's initial $150k asking price & 500TB offer)
  - Specific contingency mechanics (Financial penalty clauses & SLA credits details)
  - Speaker identity, tone, and evidentiary transcript traceability


The raw 1,152-character procurement discussion was compressed into a clean 279-character JSON state object—slashing context overhead by 75.8%.

Essential decision variables (for example the $125k budget, 350TB storage target, 99.99% SLA, and AES-256 encryption requirement) were preserved in a structured, machine-readable format.

The output also makes the trade-off explicit: semantic compression is lossy. Conversational tone, negotiation history, and transcript-level traceability were intentionally sacrificed to achieve higher efficiency.

Semantic Compression distills sprawling multi-turn conversations into dense state schemas. It can drastically cut token costs while preserving the most important facts for downstream steps, but it should not be treated as a perfect substitute for the original source material.

## 9. Strategy 8 — RAG / Vector Retrieval

**Goal:** Prevent context confusion and "Lost-in-the-Middle" phenomena by dynamically retrieving only high-relevance chunks from a vector store rather than flooding the context window with an entire document corpus.

**How it works:** Documents are divided into semantic chunks with attached metadata. When a query arrives, vector similarity scoring ranks the most relevant chunks. A focused retrieved context can outperform dumping a large unparsed corpus into the prompt, but only when the retrieval step actually surfaces the right evidence.

### Chunking, Indexing, and Vector Store Setup

In [25]:
# =====================================================================
# Strategy 8 — RAG / Vector Retrieval (Part 1: Chunking & Indexing)
# =====================================================================

import numpy as np
from sentence_transformers import SentenceTransformer

# 1. Multi-document knowledge base with metadata
raw_documents = [
    {
        "doc_id": "POL-2026-01",
        "category": "Remote Work",
        "content": "SECTION 1: Home Office Support. Remote employees receive a one-time $500 stipend for ergonomics. SECTION 2: Equipment Claims. For accidental laptop or peripheral damage occurring offsite, employees must submit Form EQ-9 within 48 hours for manager approval and immediate depot replacement."
    },
    {
        "doc_id": "POL-2026-02",
        "category": "Travel",
        "content": "SECTION 1: Booking Flights. All domestic and international flights must be reserved via the corporate travel portal at least 14 days prior to departure. SECTION 2: Per Diem Limits. Daily food allowances are strictly capped at $75 USD without itemized receipts."
    },
    {
        "doc_id": "POL-2026-03",
        "category": "Security",
        "content": "SECTION 1: Password Protocols. Master passwords must exceed 16 characters and undergo mandatory 90-day rotation. SECTION 2: Endpoint Compliance. All remote laptops must execute disk encryption (AES-256) and maintain active agent check-ins every 24 hours."
    }
]

# 2. Simple Chunking Strategy: Split documents into distinct semantic passages
chunked_store = []
for doc in raw_documents:
    sections = doc["content"].split(" SECTION ")
    for idx, sec in enumerate(sections):
        if not sec.strip():
            continue
        sec_text = "SECTION " + sec if not sec.startswith("SECTION") else sec
        chunked_store.append({
            "chunk_id": f"{doc['doc_id']}_c{idx}",
            "doc_id": doc["doc_id"],
            "category": doc["category"],
            "text": sec_text.strip()
        })

# 3. Compute vector embeddings for all chunks
embedder = SentenceTransformer("all-MiniLM-L6-v2")
chunk_texts = [c["text"] for c in chunked_store]
chunk_embeddings = embedder.encode(chunk_texts, convert_to_numpy=True)

print(f"RAG Index Ready: {len(raw_documents)} policy documents split into {len(chunked_store)} distinct vector chunks.")

RAG Index Ready: 3 policy documents split into 6 distinct vector chunks.


### Vector Search & Evaluation Metrics Engine

In [26]:
# =====================================================================
# Strategy 8 — RAG / Vector Retrieval (Part 2: Search & Evaluation)
# =====================================================================

def evaluate_and_retrieve_chunks(user_query: str, top_k: int = 1, category_filter: str = None) -> list[dict]:
    """
    Encodes query, applies optional metadata filtering, and ranks chunks by similarity.
    """
    query_vec = embedder.encode(user_query, convert_to_numpy=True)
    
    # Cosine similarity scoring
    scores = np.dot(chunk_embeddings, query_vec) / (
        np.linalg.norm(chunk_embeddings, axis=1) * np.linalg.norm(query_vec)
    )
    
    # Rank indices
    ranked_indices = np.argsort(scores)[::-1]
    
    retrieved_chunks = []
    print(f"--- RAG RETRIEVAL EVALUATION FOR QUERY: '{user_query}' ---")
    
    for idx in ranked_indices:
        chunk = chunked_store[idx]
        score = scores[idx]
        
        # Apply metadata filter if provided
        if category_filter and chunk["category"] != category_filter:
            continue
            
        print(f"  • Match Score: {score:.3f} | [{chunk['chunk_id']} | {chunk['category']}] -> {chunk['text'][:75]}...")
        retrieved_chunks.append(chunk)
        if len(retrieved_chunks) == top_k:
            break
            
    return retrieved_chunks

print("Retriever and Evaluation engine ready.")

Retriever and Evaluation engine ready.


### Baseline Comparison (Full Unfiltered Corpus vs. Focused RAG)

In [27]:
# =====================================================================
# Strategy 8 — RAG / Vector Retrieval (Part 3: Baseline Comparison)
# =====================================================================

from langchain_core.messages import HumanMessage, SystemMessage

user_query = "What specific form is required if my laptop gets damaged while working remotely?"

# 1. Retrieve specific chunk via RAG
rag_chunks = evaluate_and_retrieve_chunks(user_query, top_k=1)
rag_context = rag_chunks[0]["text"]

# 2. Build full unparsed corpus baseline
full_corpus_context = "\n\n".join([d["content"] for d in raw_documents])

# --- TEST A: FULL CORPUS BASELINE ---
prompt_full = [
    SystemMessage(content="You are an HR/IT Policy Assistant. Answer the query using the context."),
    HumanMessage(content=f"QUERY: {user_query}\n\nFULL CORPUS CONTEXT:\n{full_corpus_context}")
]
response_full = llm.invoke(prompt_full)

# --- TEST B: FOCUSED RAG INJECTION ---
prompt_rag = [
    SystemMessage(content="You are an HR/IT Policy Assistant. Answer the query using the context."),
    HumanMessage(content=f"QUERY: {user_query}\n\nRETRIEVED RAG PASSAGE:\n{rag_context}")
]
response_rag = llm.invoke(prompt_rag)


# --- DISPLAY COMPARISON METRICS ---
print("\n" + "=" * 70)
print(" 1. CONTEXT OVERHEAD & DENSITY COMPARISON")
print("=" * 70)
print(f"Full Corpus Context Size : {len(full_corpus_context)} characters")
print(f"Focused RAG Chunk Size   : {len(rag_context)} characters")
print(f"Context Reduction        : {((len(full_corpus_context) - len(rag_context)) / len(full_corpus_context)) * 100:.1f}% reduction")

print("\n" + "=" * 70)
print(" 2. AGENT RESPONSE COMPARISON")
print("=" * 70)
print(f"--- Baseline Response (Full Unparsed Corpus) ---\n{response_full.content}\n")
print(f"--- RAG Response (Focused Vector Chunk) ---\n{response_rag.content}")

--- RAG RETRIEVAL EVALUATION FOR QUERY: 'What specific form is required if my laptop gets damaged while working remotely?' ---
  • Match Score: 0.534 | [POL-2026-03_c1 | Security] -> SECTION 2: Endpoint Compliance. All remote laptops must execute disk encryp...

 1. CONTEXT OVERHEAD & DENSITY COMPARISON
Full Corpus Context Size : 807 characters
Focused RAG Chunk Size   : 141 characters
Context Reduction        : 82.5% reduction

 2. AGENT RESPONSE COMPARISON
--- Baseline Response (Full Unparsed Corpus) ---
Based on the provided context, if your laptop gets damaged while working remotely, you are required to submit Form EQ-9 within 48 hours of the incident. This form must be approved by your manager, and upon approval, you will receive an immediate depot replacement for your damaged laptop.

--- RAG Response (Focused Vector Chunk) ---
Based on the provided context, there is no specific information about a required form for reporting a damaged laptop while working remotely. The retrieved

The embedding model ranked passage POL-2026-03_c1 (Security Endpoint Compliance) as the top similarity match (Score: 0.534), cutting context overhead by 82.5%.

Notice how the Full Corpus Baseline found the correct answer (Form EQ-9 from the Remote Work Policy), while the RAG Response reported no information. Why? The vector retriever selected a *security chunk* instead of the *equipment repair chunk* because both discussed "remote laptops."

***This result clearly demonstrates the core challenge of real-world RAG:*** retrieval accuracy directly dictates generation accuracy. If the retrieval step selects the wrong chunk, the LLM will correctly refuse to hallucinate based on its limited context window.

RAG drastically reduces prompt bloat, but its success depends heavily on robust chunking, metadata filtering, and retrieval quality.



## 10. Strategy 9 — Sub-agent Architectures (Context Quarantine)

**Goal:** Eliminate Context Clash and multi-domain confusion by delegating sub-tasks to isolated worker agents governed by an Orchestrator and Lead Resolver.

**How it works:**

- Orchestrator: Dynamically decomposes a complex prompt and delegates independent research directives to specialized worker agents.
- Context Quarantine: Each worker executes inside its own isolated context window. Their raw conversation threads never mix.
- Conflict Resolution: The Lead Agent reviews worker outputs specifically to detect and resolve domain conflicts (e.g., Finance budget vs. Security compliance requirements) to make a unified decision.


### Building the Orchestrator and Isolated Worker Agents

In [28]:
# =====================================================================
# Strategy 9 — Sub-agent Architecture (Part 1: Orchestrator & Workers)
# =====================================================================

import json
from langchain_core.messages import HumanMessage, SystemMessage

# Complex multi-domain vendor evaluation request
complex_user_request = """
Evaluate renewing our enterprise telemetry contract with DataFlow Inc.:
- Legal context: Data centers are hosted in Frankfurt (GDPR compliant), but vendor requests a 3-year binding commitment without an early termination clause.
- Security context: Vendor experienced a minor breach last year; they patched it and have SOC2 Type II, but do not yet support mandatory SAML 2.0 SSO.
- Finance context: Base price is $140,000/yr. Our hard annual budget cap is $120,000/yr. Vendor offers a 15% discount if paid upfront annually.
"""

# 1. ORCHESTRATOR NODE: Decomposes request into targeted sub-agent tasks
def orchestrator_agent(user_request: str) -> list[dict]:
    orchestrator_prompt = [
        SystemMessage(content=(
            "You are an Agent Coordinator. Decompose the user's request into distinct domain tasks.\n"
            "Return a JSON array of objects, where each object has:\n"
            "- 'role': Domain name (e.g., 'Legal', 'Security', 'Finance')\n"
            "- 'task': Specific evaluation instruction for that worker.\n"
            "Output ONLY valid JSON."
        )),
        HumanMessage(content=user_request)
    ]
    
    response = llm.invoke(orchestrator_prompt)
    clean_json = response.content.strip().replace("```json", "").replace("```", "").strip()
    return json.loads(clean_json)

# 2. ISOLATED WORKER AGENT EXECUTOR
def execute_quarantined_worker(role: str, task: str) -> dict:
    """Executes a worker in a completely isolated context thread."""
    worker_prompts = {
        "Legal": "You are a Senior Legal Counsel. Evaluate terms, data residency, and contract commitment risks.",
        "Security": "You are a Chief Information Security Officer (CISO). Evaluate security compliance, vulnerabilities, and SSO requirements.",
        "Finance": "You are a Corporate Finance Director. Evaluate pricing, discounts, and strict budget caps."
    }
    
    system_instruction = worker_prompts.get(role, f"You are a specialized {role} expert.")
    
    # Quarantined context thread (fresh, unpolluted message list)
    quarantined_thread = [
        SystemMessage(content=f"{system_instruction} Give a concise recommendation (APPROVE, CONDITIONAL, or REJECT) with rationale."),
        HumanMessage(content=f"TASK: {task}")
    ]
    
    response = llm.invoke(quarantined_thread)
    return {"role": role, "finding": response.content.strip()}

print("Orchestrator and Quarantined Worker framework ready.")

Orchestrator and Quarantined Worker framework ready.


### Orchestration Execution and Parallel Quarantine Run

In [29]:
# =====================================================================
# Strategy 9 — Sub-agent Architecture (Part 2: Orchestrated Workflow)
# =====================================================================

# Step 1: Orchestrator dynamically decomposes the task
subtask_plan = orchestrator_agent(complex_user_request)

print("=" * 70)
print(" 1. ORCHESTRATOR TASK DECOMPOSITION")
print("=" * 70)
for item in subtask_plan:
    print(f"  • Assigned [{item['role']} Worker]: {item['task']}")

# Step 2: Execute sub-agents in quarantined, isolated contexts
worker_results = []
print("\n" + "=" * 70)
print(" 2. QUARANTINED WORKER EXECUTION (ISOLATED THREADS)")
print("=" * 70)

for item in subtask_plan:
    result = execute_quarantined_worker(item['role'], item['task'])
    worker_results.append(result)
    print(f"\n✓ [{result['role']} Worker Finished - Context Isolated]")
    print(f"  Output: {result['finding'][:150]}...")

 1. ORCHESTRATOR TASK DECOMPOSITION
  • Assigned [Legal Worker]: Review the 3-year binding commitment clause and assess the implications of lacking an early termination clause, especially in the context of GDPR compliance for data hosted in Frankfurt.
  • Assigned [Security Worker]: Evaluate the vendor's security posture, including the recent minor breach, SOC2 Type II certification, and the absence of mandatory SAML 2.0 SSO support. Assess the risk and potential mitigation strategies.
  • Assigned [Finance Worker]: Analyze the financial aspects of the contract, including the base price of $140,000/yr, the hard annual budget cap of $120,000/yr, and the 15% discount for upfront annual payment. Determine the financial feasibility and potential cost-saving strategies.

 2. QUARANTINED WORKER EXECUTION (ISOLATED THREADS)

✓ [Legal Worker Finished - Context Isolated]
  Output: To provide a comprehensive evaluation, I would need to review the specific terms of the 3-year binding commitment c

### Conflict Detection & Resolution vs. Single-Agent Baseline

In [30]:
# =====================================================================
# Strategy 9 — Sub-agent Architecture (Part 3: Conflict Resolution & Baseline)
# =====================================================================

# Step 3: LEAD RESOLVER AGENT (Conflict Detection & Handoff Synthesis)
def lead_conflict_resolver(user_request: str, worker_reports: list[dict]) -> str:
    formatted_reports = "\n\n".join([f"=== {r['role'].upper()} WORKER REPORT ===\n{r['finding']}" for r in worker_reports])
    
    resolver_prompt = [
        SystemMessage(content=(
            "You are the Chief Executive Decision Lead. Review worker findings to:\n"
            "1. Detect any explicitly CONFLICTING recommendations between domain workers (e.g., Finance budget approval vs Security SSO rejection).\n"
            "2. Resolve the conflicts by establishing binding conditions.\n"
            "3. Issue the final GO / NO-GO executive directive."
        )),
        HumanMessage(content=f"ORIGINAL REQUEST:\n{user_request}\n\nQUARANTINED WORKER REPORTS:\n{formatted_reports}")
    ]
    
    response = llm.invoke(resolver_prompt)
    return response.content.strip()

# Execute Lead Resolver
final_resolution = lead_conflict_resolver(complex_user_request, worker_results)

# --- TEST BASELINE: SINGLE ALL-IN-ONE AGENT (Without Context Quarantine) ---
single_agent_prompt = [
    SystemMessage(content="You are an Enterprise Reviewer. Evaluate Legal, Security, and Finance simultaneously and make a decision."),
    HumanMessage(content=complex_user_request)
]
single_agent_response = llm.invoke(single_agent_prompt)


# --- DISPLAY DIRECT MODEL COMPARISON ---
print("=" * 70)
print(" 1. MULTI-AGENT ARCHITECTURE (ORCHESTRATED & RESOLVED)")
print("=" * 70)
print(final_resolution)

print("\n" + "=" * 70)
print(" 2. SINGLE-AGENT BASELINE (UNQUARANTINED)")
print("=" * 70)
print(single_agent_response.content)

 1. MULTI-AGENT ARCHITECTURE (ORCHESTRATED & RESOLVED)
EXECUTIVE DECISION:

After reviewing the reports from the Legal, Security, and Finance workers, I have identified a conflict between the Finance and Security recommendations. The Finance worker has recommended rejecting the contract due to the base price exceeding the hard annual budget cap of $120,000, even with the 15% discount. On the other hand, the Security worker has recommended a conditional approval, subject to certain conditions being met.

To resolve this conflict, I propose the following binding conditions:

1. The vendor must agree to reduce the base price to align with the $120,000 budget cap. This can be achieved through further negotiation or by exploring alternative pricing models that fit within the allocated budget.

2. The vendor must provide a detailed incident response plan that aligns with our organization's requirements and demonstrates a proven track record of effectively managing security incidents.

3. The

The **Orchestrated Multi-Agent Decision** shows the structural benefit of context quarantine: each worker reasoned within a narrower domain-specific context, and a lead resolver then synthesized the worker outputs into a single decision.

The Single-Agent Baseline is also fairly coherent in this example, so this run should not be read as proof that multi-agent is always smarter. The more defensible takeaway is narrower: sub-agent isolation can make decomposition and synthesis easier to manage when different parts of a task would otherwise compete for the same context window.

Quarantining each domain worker reduces the chance that Legal, Security, and Finance concerns will blur together during intermediate reasoning. The lead resolver still has to reconcile conflicts carefully, and its synthesis can introduce its own mistakes if it is not well grounded.

Sub-agent Architectures and Context Quarantine are most useful when subtasks are relatively independent and the main problem is context interference. They are not a blanket replacement for a well-scoped single agent.

## Summary & Best Practices

Managing context effectively is the single most critical factor in building reliable, cost-effective LLM agents. 

### Quick Selection Matrix:
- **For simple chat applications:** Use **FIFO (Strategy 1)** or **Compaction (Strategy 2)**.
- **For agents with 10+ tools:** Use **Dynamic Tool Selection (Strategy 3)** to prevent tool bloat.
- **For noisy tool/API outputs:** Use **Context Pruning (Strategy 4)** to remove junk text before main agent execution.
- **For long, multi-step workflows:** Combine **LangGraph State Offloading (Strategy 5)** or **Filesystem Scratchpads (Strategy 6)**.
- **For large document repositories:** Use **RAG (Strategy 8)**.
- **For complex multi-domain problems:** Use **Sub-agent Architectures (Strategy 9)** to quarantine context threads.

## 12. Conclusion

This notebook demonstrated all nine context management strategies from `context_management.md` using real Granite model calls on watsonx.ai.

Key themes across all strategies:

- **Context is not free.** Every token influences the model's output. Treat context budget like memory budget.
- **Know your failure mode.** Poisoning, Distraction, Confusion, and Clash require different mitigations.
- **Information gathering and decision making should be separated.** Sub-agents work well for the former; single agents work better for the latter.
- **Summarization loses information.** Offloading to files or a scratchpad (Strategies 5 and 6) is safer.
- **Benchmarks understate the problem.** Test at your actual context lengths with actual distractors.
- **Structure your context.** Typed LangGraph state (as used in Strategy 5) makes pruning, summarization, and selective retrieval dramatically easier.

### Where to go next

- [Tool RAG recipe](../ToolRAG/ToolRAG_Agent.ipynb) — production-grade embedding-based tool selection (Strategy 3)
- [Function Calling Agent recipe](../Function_Calling/Function_Calling_Agent.ipynb) — building LangGraph agents from scratch
- [Tracing recipe](../Tracing/Tracing_Agent.ipynb) — observing context usage across agent runs
